In [ ]:
import neurokit2 as nk
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from matplotlib import pyplot as plt

In [ ]:
mode = 0
"""
mode = 0 ---> non mace
mode = 1 ---> mace
"""
if mode == 0:
    save_win_file_name = 'non_mace'
elif mode == 1:
    save_win_file_name = 'mace'
Zscore = True

In [ ]:
if mode == 0:
    file_path = r'V:\dunwei\MACE\dataset\SKNA_signal\conversion_data\healthy/'
elif mode == 1:
    file_path = r'V:\dunwei\MACE\dataset\SKNA_signal\conversion_data\patient/'

hp_path = r'V:\dunwei\MACE\dataset\SKNA_signal/'

if mode == 0:
    if Zscore is True:
        save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal\non_mace_zscore/'
    else:
        save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal\non_mace/'
elif mode == 1:
    if Zscore is True:
        save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal\mace_zscore/'
    else:
        save_path = r'V:\dunwei\MACE\dataset\ECG_signal\ch1\sr500_0.5_50_10s_ECG_signal\mace/'

In [ ]:
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f'Create directory: {save_path}')
else:
    print(f'Directory already exists: {save_path}')

In [ ]:
original_sr = 10000
resample_rate = 500
window_size = 10 # unit: second
highcut = 50
lowcut = 0.5

In [ ]:
if mode == 0:
    hp_id = pd.read_csv(os.path.join(hp_path, 'MACE_h_id.csv'))
    raw_id = hp_id['research_ID'].values
    covert_id = hp_id['conversion_ID'].values
elif mode == 1:
    hp_id = pd.read_csv(os.path.join(hp_path, 'MACE_p_id.csv'))
    raw_id = hp_id['research_ID'].values
    covert_id = hp_id['conversion_ID'].values

In [ ]:
win_num_dict = {'research_ID': [], 'window_num': [], 'time(s)': []}

for idx, id in enumerate(tqdm(raw_id, desc='Processing IDs')):
    segment_nor_list = []
    segment_list = []
    try:
        signal = pd.read_csv(os.path.join(file_path, f'{id}.csv'))
        ch1_signal = signal['Ch1'].values.astype(np.float32)
        ecg_signal_detrended = nk.signal_detrend(ch1_signal, method = 'polynomial', order=0, sampling_rate=original_sr)
        ecg_signal_filter_notch = nk.signal_filter(ecg_signal_detrended, sampling_rate=original_sr, method="powerline", powerline=60)
        ecg_signal_filter = nk.signal_filter(ecg_signal_filter_notch, sampling_rate=original_sr, lowcut=lowcut, highcut=highcut, method="butterworth", order=2)
        ecg_signal_resampled = nk.signal_resample(ecg_signal_filter, sampling_rate=original_sr, desired_sampling_rate=resample_rate)

        time = ecg_signal_resampled.shape[0] / resample_rate
        win_num = int(len(ecg_signal_resampled) / (window_size * resample_rate))
        win_num_dict['research_ID'].append(str(id))
        win_num_dict['window_num'].append(int(win_num))
        win_num_dict['time(s)'].append(float(time))

        for i in range(win_num):
            start = i * window_size * resample_rate
            end = (i + 1) * window_size * resample_rate
            if end > len(ecg_signal_resampled):
                continue
            segment = ecg_signal_resampled[start:end]

            if Zscore is True:
                segment_mean = np.mean(segment)
                segment_std = np.std(segment)
                segment_nor = (segment - segment_mean) / segment_std
                segment_nor_list.append(segment_nor)
            else:
                segment_list.append(segment)

        if Zscore is True:
            segment_nor_array = np.array(segment_nor_list).astype(np.float32)
            if mode == 0:
                label_arr = np.zeros((segment_nor_array.shape[0], 1), dtype=np.float32)
            elif mode == 1:
                label_arr = np.ones((segment_nor_array.shape[0], 1), dtype=np.float32)
            else:
                raise ValueError("Mode should be 0 or 1.")

            segment_ECG_nor_array = np.hstack((label_arr, segment_nor_array))
            np.save(os.path.join(save_path, f'{covert_id[idx]}.npy'), segment_ECG_nor_array)

        else:
            segment_array = np.array(segment_list).astype(np.float32)
            if mode == 0:
                label_arr = np.zeros((segment_array.shape[0], 1), dtype=np.float32)
            elif mode == 1:
                label_arr = np.ones((segment_array.shape[0], 1), dtype=np.float32)
            else:
                raise ValueError("Mode should be 0 or 1.")

            segment_ECG_array = np.hstack((label_arr, segment_array))
            np.save(os.path.join(save_path, f'{covert_id[idx]}.npy'), segment_ECG_array)


    except Exception as e:
        print(f"Error processing ID {id}: {e}")
        continue
    

result_df = pd.DataFrame(win_num_dict)
result_df.to_csv(os.path.join(save_path, f'{save_win_file_name}_window_num.csv'), index=False)